# Digit Classifier Demo

This notebook loads the trained CNN and runs inference on the original sample image `3-digit.PNG` from the `assets/` folder.
The code below uses the saved checkpoint `checkpoints/model_best.pth` and the same MNIST preprocessing used during training.

In [ ]:
import os
import sys
import torch
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

# Add project root to path (works from notebooks/ directory)
project_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.model import SimpleCNN

# Load the sample image using a relative path
img_path = os.path.join('..', 'assets', '3-digit.PNG')
img = Image.open(img_path).convert('L')

# Load the trained model
model = SimpleCNN()
ckpt_path = os.path.join('..', 'checkpoints', 'model_best.pth')
ckpt = torch.load(ckpt_path, map_location='cpu')
model.load_state_dict(ckpt['model_state'])
model.eval()

# Preprocess the image
preprocess = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
x = preprocess(img).unsqueeze(0)

# Run inference
with torch.no_grad():
    logits = model(x)
    probs = torch.softmax(logits, dim=1)[0]
    pred = int(logits.argmax(dim=1).item())

# Display results
top5 = sorted([(float(p), i) for i, p in enumerate(probs)], reverse=True)[:5]
print('Input image:', img_path)
print('Predicted digit:', pred)
print('Top 5 probabilities:')
for conf, cls in top5:
    print(f'  class {cls}: {conf:.6f}')

plt.figure(figsize=(3, 3))
plt.imshow(img, cmap='gray')
plt.title(f'Predicted class: {pred}')
plt.axis('off')
plt.show()
